# Finite Elements — Detailed Summary

## L1: Introduction

### Model Problem — Poisson's Equation
On $\Omega=[0,1]^2$: find $u$ such that
$$-\nabla^2 u = f, \quad u(0,y)=u(1,y)=0\;(\text{Dirichlet}),\quad \frac{\partial u}{\partial y}(x,0)=\frac{\partial u}{\partial y}(x,1)=0\;(\text{Neumann}).$$

### Triangulations
**Definition.** A triangulation $\mathcal{T}$ of polygonal $\Omega\subset\mathbb{R}^2$ is a set of triangles $\{K_i\}$ s.t.:
1. $\mathrm{int}\,K_i\cap\mathrm{int}\,K_j=\emptyset$ for $i\neq j$ (no overlaps)
2. $\cup K_i=\bar{\Omega}$ (covers $\Omega$)
3. No vertex of any triangle lies in the interior of an edge of another (no hanging nodes)

### P1 Finite Element Space
$$V_h = \{v\in C^0(\Omega): v|_{K_i}\text{ is linear }\forall K_i\in\mathcal{T}\}$$
$$\mathring{V}_h = \{v\in V_h: v(0,y)=v(1,y)=0\} \quad(\text{incorporates Dirichlet BCs})$$

### $L^2$ Norm and Space
$$\|f\|_{L^2(\Omega)}=\left(\int_\Omega|f(x)|^2\,dx\right)^{1/2}, \qquad L^2(\Omega)=\{f:\|f\|_{L^2}<\infty\}$$
Two functions identified if $\|f-g\|_{L^2}=0$.

### Finite Element Derivative
Defined in $L^2$: $\frac{\partial^{FE}u}{\partial x_i}\big|_K = \frac{\partial u}{\partial x_i}\big|_K$ for each $K$. Not uniquely defined on edges, but unique in $L^2$.

### Deriving the Weak Form
1. Multiply $-\nabla^2 u=f$ by test function $v\in\mathring{V}_h$, integrate
2. Integration by parts on each triangle:
$$\sum_i\left(\int_{K_i}\nabla v\cdot\nabla u\,dx - \int_{\partial K_i}v\,n\cdot\nabla u\,dS\right) = \int_\Omega vf\,dx$$
3. Interior edge contributions cancel ($n^-=-n^+$)
4. Boundary terms vanish: $v=0$ on Dirichlet, $n\cdot\nabla u=0$ on Neumann
5. **FE approximation**: find $u_h\in\mathring{V}_h$ s.t.
$$\int_\Omega\nabla v\cdot\nabla u_h\,dx = \int_\Omega vf\,dx, \quad\forall v\in\mathring{V}_h$$

### Practical Implementation
**Nodal basis**: $\phi_i(z_j)=\delta_{ij}$ for vertices $z_j$.

Expanding $u_h=\sum_j u_j\phi_j$, $v=\sum_i v_i\phi_i$ gives the matrix system $K\mathbf{u}=\mathbf{f}$:
$$K_{ij}=\int_\Omega\nabla\phi_i\cdot\nabla\phi_j\,dx, \qquad f_i=\int_\Omega\phi_i f\,dx$$
$K$ is sparse (most $\phi_i,\phi_j$ have non-overlapping support). Assembly by looping over elements.

## L2: Finite Element Spaces — Local to Global

### Ciarlet's Finite Element
**Definition.** A triple $(K,\mathcal{P},\mathcal{N})$ where:
1. $K\subset\mathbb{R}^n$: bounded closed domain (polygon)
2. $\mathcal{P}$: finite-dimensional space of shape functions on $K$
3. $\mathcal{N}=(N_0,\ldots,N_k)$: basis for dual space $\mathcal{P}'$

Examples of dual functions: point evaluation, line integrals, area integrals, derivative evaluation.

### Nodal Basis
**Definition.** $\{\phi_j\}$ with $N_i(\phi_j)=\delta_{ij}$.

### Vandermonde Matrix
Given arbitrary basis $\{\psi_i\}$: $V_{ij}=N_j(\psi_i)$.

**Lemma.** Nodal basis coefficients: $\phi_i=\sum_j\mu_{ij}\psi_j$ where $\mu=V^{-1}$.

### Unisolvence (Dual Condition Lemma)
The following are equivalent:
1. $\mathcal{N}$ is a basis for $\mathcal{P}'$
2. Vandermonde matrix $V$ is invertible
3. ($\mathcal{N}$ determines $\mathcal{P}$): $N_i(v)=0\;\forall i\Rightarrow v\equiv 0$

### 1D Lagrange Element of Degree $k$
- $K=[a,b]$, $\mathcal{P}=$ polynomials of degree $\leq k$, $N_i(v)=v(x_i)$ at $k+1$ equispaced points.
- Nodal basis: $\phi_i(x)=\prod_{j\neq i}\frac{x-x_j}{x_i-x_j}$
- **Corollary**: Unisolvent (vanishing at $k+1$ points $\Rightarrow$ zero, by fund. thm of algebra).

### Triangular Lagrange Elements (Pk)
- $K$: triangle with vertices $z_1,z_2,z_3$
- $\mathcal{P}$: degree $k$ polynomials on $K$
- Nodes: $x_{i,j}=z_1+(z_2-z_1)\frac{i}{k}+(z_3-z_1)\frac{j}{k}$

**P1 unisolvence proof**: $p$ linear vanishing at 3 vertices $\Rightarrow$ $p|_{\Pi_1}$ vanishes at 2 pts $\Rightarrow$ $p=cL_1$ $\Rightarrow$ $p(z_1)=cL_1(z_1)=0\Rightarrow c=0$.

**P2 unisolvence proof**: $p=0$ at 3 pts on each edge $\Rightarrow$ $p=L_1Q_1$, then $Q_1=0$ on $\Pi_2$ $\Rightarrow$ $p=cL_1L_2$, midpoint condition $\Rightarrow$ $c=0$.

**Key lemma for 2D**: If polynomial $p$ vanishes on hyperplane $\Pi_L=\{x:L(x)=0\}$, then $p=LQ$ with $\deg(Q)=\deg(p)-1$.

### Exotic Elements
- **Cubic Hermite**: $K$=triangle, $\mathcal{P}$=cubics, $\mathcal{N}$=vertex values + vertex gradients + centre value (10 DOF)
- **Quintic Argyris**: $K$=triangle, $\mathcal{P}$=quintics, $\mathcal{N}$=vertex values, gradients, Hessians + edge normal derivatives (21 DOF). Has $C^1$ geometric decomposition.

### Geometric Decomposition & Global Spaces

**Continuity Lemma.** $V$ is $C^m$ iff functions and derivatives up to degree $m$ agree at all shared vertices and edges.

**$C^m$ geometric decomposition**: each nodal variable assigned to a mesh entity (vertex/edge/cell); the sub-element $(w,\mathcal{P}|_w,\mathcal{N}_w)$ is itself a FE for each entity $w$.

- Lagrange Pk: has $C^0$ geometric decomposition
- Argyris: has $C^1$ geometric decomposition

**Global $C^m$ space**: Match nodal variables on shared mesh entities between adjacent elements $\Rightarrow$ $C^m$ continuity across triangulation.

## L3: Interpolation Operators

### Local Interpolator
$$\mathcal{I}_Kv(x) = \sum_{i=0}^k N_i(v)\phi_i(x)$$

**Properties:**
1. Linear ($\mathcal{I}_K$ is a linear operator)
2. Preserves nodal values: $N_i(\mathcal{I}_Kv)=N_i(v)$
3. Projection: $\mathcal{I}_Kp=p$ for all $p\in\mathcal{P}$

### Global Interpolator
$\mathcal{I}_hu|_K = \mathcal{I}_Ku$ for each $K\in\mathcal{T}_h$.

### Sobolev's Inequality (for continuous functions)
For $k>n/2$: $\|u\|_{C^\infty(\Omega)}\leq C\|u\|_{H^k(\Omega)}$.

For derivatives: $k-m>n/2$: $\|u\|_{C^{m,\infty}}\leq C\|u\|_{H^k}$.

### Averaged Taylor Polynomial
**Definition.** For $f\in C^{k,\infty}$, domain star-shaped w.r.t. ball $B$:
$$Q_{k,B}f(x) = \frac{1}{|B|}\int_B T^kf(y,x)\,dy$$

**Key property:** $D^\beta Q_{k,B}f = Q_{k-|\beta|,B}D^\beta f$.

### Taylor Polynomial Error Estimate
$$\|D^\beta(f-Q_{k,B}f)\|_{L^2(\Omega)}\leq C\frac{|\Omega|^{1/2}}{|B|^{1/2}}d^{k+1-|\beta|}|f|_{H^{k+1}(\Omega)}$$

### Interpolation Error Chain

1. **Bound on $\mathcal{I}_{K_1}$** (diameter 1 triangle): $\|\mathcal{I}_{K_1}u\|_{H^k}\leq C\|u\|_{H^k}$
   - Proof: triangle inequality on nodal basis, Sobolev inequality

2. **Local error** (diameter 1): $|\mathcal{I}_{K_1}u-u|_{H^i}\leq C|u|_{H^{k+1}}$
   - Proof: add/subtract $Q_{k,B}u$; use projection property $\mathcal{I}_K Q_{k,B}u=Q_{k,B}u$

3. **Scaling** (diameter $d$): $|\mathcal{I}_Ku-u|_{H^i(K)}\leq C_Kd^{k+1-i}|u|_{H^{k+1}(K)}$
   - Proof: change of variables $x\to x/d$, collect powers of $d$

4. **Global estimate**: $\|\mathcal{I}_hu-u\|_{H^i(\Omega)}\leq Ch^{k+1-i}|u|_{H^{k+1}(\Omega)}$
   - Proof: sum local estimates, use $d_K\leq h$, need positive minimum aspect ratio

## L4: Finite Element Problems — Solvability & Stability

### Hilbert Space Foundations

| Concept | Definition |
|---------|----------|
| Vector space | $V$ with $+$ and $\times$, closed, zero element |
| Bilinear form | $b:V\times V\to\mathbb{R}$, linear in each argument |
| Inner product | Symmetric bilinear form with $(v,v)\geq 0$ and $(v,v)=0\iff v=0$ |
| $L^2$ inner product | $(f,g)_{L^2}=\int_\Omega fg\,dx$ |
| $H^1$ inner product | $(f,g)_{H^1}=\int_\Omega fg+\nabla f\cdot\nabla g\,dx$ |
| Norm | $\|v\|=\sqrt{(v,v)}$ for inner product space |
| Complete | Every Cauchy sequence has a limit in $V$ |
| Hilbert space | Complete inner product space |

All finite-dimensional inner product spaces are complete $\Rightarrow$ FE spaces with $L^2$ or $H^1$ are Hilbert.

### Linear Functionals
- $L:H\to\mathbb{R}$ bounded iff $|L(u)|\leq C\|u\|_H$
- Bounded $\iff$ continuous (for linear functionals)
- **Dual space** $H'$: space of continuous linear functionals
- **Dual norm**: $\|L\|_{H'}=\sup_{0\neq v}\frac{L(v)}{\|v\|_H}$
- **Riesz Representation**: $\forall L\in H'$, $\exists u\in H$ s.t. $L(v)=(u,v)$ and $\|u\|_H=\|L\|_{H'}$

### Linear Variational Problem
Find $u\in V$ s.t. $b(u,v)=F(v)$ $\forall v\in V$.

### Continuity & Coercivity
- **Continuous**: $|b(u,v)|\leq M\|u\|_V\|v\|_V$
- **Coercive**: $b(u,u)\geq\gamma\|u\|_V^2$

### Lax-Milgram Theorem
If $b$ continuous and coercive, $F$ continuous, then $\exists!$ $u\in V$ with $\|u\|_V\leq\frac{1}{\gamma}\|F\|_{V'}$.

### Stability
A sequence of FE discretisations is **stable** if $\gamma_h\to\gamma>0$ and $\|F\|_{V_h'}\to c<\infty$ as $h\to 0$.

### Helmholtz Problem
$b(u,v)=(u,v)_{H^1}$. Continuity: $|b(u,v)|= |(u,v)_{H^1}|\leq\|u\|_{H^1}\|v\|_{H^1}$ (Schwarz). Coercivity: $b(u,u)=\|u\|_{H^1}^2$. Both constants $=1$, $h$-independent $\Rightarrow$ stable.

### Poisson (Neumann) — Mean Estimate
For $u\in C^0$ FE space: $\|u-\bar{u}\|_{L^2}\leq C|u|_{H^1}$ where $\bar{u}=\frac{\int u}{\int 1}$.

On $\bar{V}_h=\{u:\bar{u}=0\}$: $\|u\|_{L^2}^2\leq C^2|u|_{H^1}^2$ $\Rightarrow$ $\|u\|_{H^1}^2\leq(1+C^2)b(u,u)$ $\Rightarrow$ coercive.

### Poisson (Dirichlet) — Poincaré Inequality
Needs: FE divergence theorem, trace theorem ($\|u\|_{L^2(\partial\Omega)}\leq C\|u\|_{H^1}$), then Poincaré inequality for functions vanishing on $\Gamma_0$.

## L5: Convergence of Finite Element Approximations

### Weak Derivatives
**Definition.** $D_w^\alpha f\in L^1_{loc}$ defined by:
$$\int_\Omega\phi\,D_w^\alpha f\,dx = (-1)^{|\alpha|}\int_\Omega D^\alpha\phi\,f\,dx, \quad\forall\phi\in C_0^\infty(\Omega)$$

**Key equivalences:**
- For $u\in C^0$ FE space: FE derivative $=$ weak derivative
- For $u\in C^{|\alpha|}$: strong derivative $=$ weak derivative
- For discontinuous FE: weak $\neq$ FE derivative in general

### Sobolev Spaces
$$H^k(\Omega)=\{u\in L^1_{loc}: \|u\|_{H^k}<\infty\}$$
$H^k$ is a Hilbert space (closed). $H^l\subset H^k$ for $l>k$.

**$H=W$ theorem**: $H^k\cap C^\infty$ is dense in $H^k$.

**Sobolev's inequality** (general): $k>n/2$ $\Rightarrow$ $\|u\|_{L^\infty}\leq C\|u\|_{H^k}$, and there is a $C^0$ representative.

### Strong $\leftrightarrow$ Weak Equivalence (Helmholtz)
- $u\in H^2$ solves strong form $\Rightarrow$ solves variational form (just integrate by parts)
- $u\in H^2$ solves variational form $\Rightarrow$ solves strong form:
  1. Test with $v\in C_0^\infty$: get $\|u-\nabla^2u-f\|_{L^2}=0$ (density of $C_0^\infty$ in $L^2$)
  2. Boundary: $\int_{\partial\Omega}\frac{\partial u}{\partial n}v\,dS=0$ for arbitrary $v$ $\Rightarrow$ zero Neumann BC

### Galerkin Approximation
For $V_h\subset V=H^k$: find $u_h\in V_h$ s.t. $b(u_h,v)=F(v)$ $\forall v\in V_h$.

Since $V_h\subset V$, continuity and coercivity are inherited $\Rightarrow$ Lax-Milgram applies.

### Céa's Lemma
$$\|u-u_h\|_V\leq\frac{M}{\gamma}\min_{v\in V_h}\|u-v\|_V$$

**Proof sketch:**
1. **Galerkin orthogonality**: $b(u-u_h,v)=0$ $\forall v\in V_h$ (subtract equations)
2. For any $v\in V_h$: $\gamma\|u-u_h\|^2\leq b(u-u_h,u-u_h)=b(u-u_h,u-v)+\underbrace{b(u-u_h,v-u_h)}_{=0}\leq M\|u-u_h\|\|u-v\|$
3. Divide by $\|u-u_h\|$, minimise over $v$

### Interpolation Error in $H^k$ (extended from L3)
All L3 results extend to $H^{k+1}$ functions (replace $C^{l,\infty}$ by $W^l_\infty$, pass to limit using $H=W$).

### Convergence of Helmholtz
Choose $v=\mathcal{I}_hu$ in Céa's lemma:
$$\|u_h-u\|_{H^1}\leq\frac{M}{\gamma}\|u-\mathcal{I}_hu\|_{H^1}\leq Ch^k|u|_{H^{k+1}}$$

### Aubin-Nitsche Trick ($L^2$ estimates)
**Elliptic regularity**: On convex domain, $|w|_{H^2}\leq C\|f\|_{L^2}$ for $w-\nabla^2w=f$.

**$L^2$ estimate**: $\|u_h-u\|_{L^2}\leq Ch^{k+1}\|u\|_{H^{k+1}}$

**Proof sketch:**
1. Let $w$ solve $w-\nabla^2w=u-u_h$ (dual problem)
2. $\|u-u_h\|_{L^2}^2=b(w,u-u_h)=b(w-\mathcal{I}_hw,u-u_h)$ (Galerkin orthogonality)
3. $\leq C\|u-u_h\|_{H^1}\|w-\mathcal{I}_hw\|_{H^1}\leq Ch\cdot\|u-u_h\|_{H^1}\cdot|w|_{H^2}$
4. Elliptic regularity: $|w|_{H^2}\leq C\|u-u_h\|_{L^2}$ $\Rightarrow$ divide both sides

Thus $L^2$ error gains one power of $h$ over $H^1$ error.

## L6: Stokes Equation (Mastery)

### Strong Form
$$-2\mu\nabla\cdot\varepsilon(u)+\nabla p=f, \quad \nabla\cdot u=0, \quad \varepsilon(u)=\tfrac{1}{2}(\nabla u+\nabla u^T)$$
- $u$: velocity (vector), $p$: pressure, $\mu$: viscosity, $f$: body force
- Boundary condition: $u=0$ on $\partial\Omega$ (no-slip)
- Pressure unique up to constant $\Rightarrow$ impose $\int_\Omega p=0$

### Function Spaces
$$V=(\mathring{H}^1(\Omega))^n, \quad Q=\mathring{L}^2(\Omega)=\{p\in L^2:\int p=0\}$$

### Variational Form
Find $(u,p)\in V\times Q$ s.t.
$$a(u,v)+b(v,p)=\int_\Omega f\cdot v, \quad b(u,q)=0, \quad \forall(v,q)\in V\times Q$$
where $a(u,v)=2\mu\int_\Omega\varepsilon(u):\varepsilon(v)$, $b(v,q)=-\int_\Omega q\,\nabla\cdot v$.

### Why Lax-Milgram Fails
The combined form $c(U,W)=a(u,v)+b(v,p)+b(u,q)$ is **not coercive**: $c(U,U)=a(u,u)$ with no $p$ control. Take $v=0$ to see $c(U,U)$ can be zero for nonzero $U$.

### Inf-Sup Condition
$$\inf_{0\neq q\in Q}\sup_{0\neq v\in V}\frac{b(v,q)}{\|v\|_V\|q\|_Q}\geq\beta>0$$

Equivalent to: $\|B^*q\|_{V'}\geq\beta\|q\|_Q$ $\forall q$ ($B^*$ injective).

In finite dimensions, $B^*$ injective $\iff$ $B$ surjective (rank-nullity).

### Brezzi's Theorem
Define kernel $Z=\{u\in V:b(u,q)=0\;\forall q\}$. Assume:
1. $a$ coercive on $Z$: $a(u,u)\geq\alpha\|u\|_V^2$ $\forall u\in Z$
2. Inf-sup for $b$ with constant $\beta>0$

Then $\exists!$ $(u,p)$ with stability bounds:
$$\|u\|_V\leq\frac{1}{\alpha}\|F\|_{V'}+\frac{2M}{\alpha\beta}\|G\|_{Q'}$$
$$\|p\|_Q\leq\frac{2M}{\alpha\beta}\|F\|_{V'}+\frac{2M^2}{\alpha\beta^2}\|G\|_{Q'}$$

### Discrete Inf-Sup
For FE spaces $V_h\subset V$, $Q_h\subset Q$, need:
$$\inf_{q\in Q_h}\sup_{v\in V_h}\frac{b(v,q)}{\|v\|_V\|q\|_Q}\geq\beta_h>0$$
with $\beta_h$ independent of $h$. Not automatic from continuous inf-sup!

### Element Pairs

| Pair | $V_h$ | $Q_h$ | Inf-sup? |
|------|-------|-------|----------|
| P1-P1 | $(P1)^d$ | P1 | **NO** ($\beta_h\to 0$) |
| MINI | $(P1+B3)^d$ | P1 | Yes |
| Taylor-Hood | $(P2)^d$ | P1 | Yes |

### Fortin's Trick
If $\exists$ bounded $\Pi_h:V\to V_h$ with $b(v-\Pi_hv,q_h)=0$ $\forall v,q_h$, then discrete inf-sup holds with $\beta_h=\beta/C_\Pi$.

### Error Estimate for Mixed Problems
$$\|u_h-u\|_V\leq\frac{4MM_b}{\alpha\beta_h}E_u+\frac{M_b}{\alpha}E_p$$
$$\|p_h-p\|_Q\leq\frac{3M^2M_b}{\alpha\beta_h^2}E_u+\frac{3MM_b}{\alpha\beta_h}E_p$$
where $E_u=\inf_{u_I\in V_h}\|u-u_I\|_V$, $E_p=\inf_{p_I\in Q_h}\|p-p_I\|_Q$.